# Stage 6 Materials Proxy: Application Design Tables

This notebook writes application-facing planning tables for
comparing candidate optical cases. It does not start capsule,
weld-feature, hexagon, polygonal, or discrete N-fold studies.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

import bessel_twin_core as bt
from Publication_Study import publication_diagnostics as pdiag
from vbb_study import setup_study, vbb_materials
from vbb_study.publication import materials as material_schema

PATHS = setup_study.bootstrap(Path.cwd())
CSV_OUT = PATHS["csv"] / "materials"
CSV_OUT.mkdir(parents=True, exist_ok=True)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply material
# parameter overrides for downstream analysis cells.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='materials',
    # ── edit these for materials analysis ───────────────────────────────────
    pulse_energy_uJ=10.0,
    threshold_fluence_J_cm2=1.0,
    pulse_count=1,
)

# Wire control parameters to named variables used by downstream cells.
_p = NOTEBOOK_CONTROLS.parameters or {}
PULSE_ENERGY_uJ = float(_p.get("pulse_energy_uJ", 10.0))
THRESHOLD_FLUENCE_J_CM2 = float(_p.get("threshold_fluence_J_cm2", 1.0))
PULSE_COUNT = int(_p.get("pulse_count", 1))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [ ]:
# Interactive beam quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved and this is independent of the
# material analysis cells below.
from dataclasses import replace
from vbb_study.publication import notebook_widgets as nbw

_ql_base = bt.default_config("fast")
_panel = nbw.interactive_quicklook(_ql_base, method='holographic', preset='fast')
display(_panel)


In [2]:
summary_path = CSV_OUT / "material_proxy_fluence_threshold_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
else:
    summary, _cases = vbb_materials.build_shortlist_design_table(
        pdiag.DEFAULT_SHORTLIST,
        preset="fast",
        path="realistic",
    )
    summary = material_schema.ordered_material_frame(summary.to_dict("records"))
    summary.to_csv(summary_path, index=False)


## Design Comparison

Rows are ranked by optical fluence margin and kept in the
planning_proxy category. The rank is a design-planning aid, not a
prediction of material modification success.


In [3]:
design = vbb_materials.application_design_table_from_proxy_summary(summary)
design["source_optical_route"] = design["beam_family"].map({
    "scalar_bessel": "scalar Bessel optical metric",
    "vortex_bessel": "vortex Bessel optical metric",
}).fillna(design["beam_family"])
design["qa_gate"] = design.apply(
    lambda row: "compare_proxy_only"
    if row["material_model_status"] == "planning_proxy"
    and row["calibration_status"] == "uncalibrated"
    else "review_required",
    axis=1,
)
design = material_schema.ordered_material_frame(design.to_dict("records"))

design_path = CSV_OUT / "material_application_design_table.csv"
design.to_csv(design_path, index=False)

display_cols = [
    "planning_rank",
    "case_id",
    "beam_family",
    "qa_gate",
    "material_model_status",
    "calibration_status",
    "fluence_to_threshold_ratio",
    "thresholded_equivalent_diameter_um",
    "xz_proxy_length_um",
    "xz_energy_conservation_status",
]
display(design[display_cols])
print(design_path)


,planning_rank,case_id,beam_family,qa_gate,material_model_status,calibration_status,fluence_to_threshold_ratio,thresholded_equivalent_diameter_um,xz_proxy_length_um,xz_energy_conservation_status
0,1,ell0_core3_L150,scalar_bessel,compare_proxy_only,planning_proxy,uncalibrated,31.374000,7.537376,375.0,normalised_visualisation
1,2,ell5_core4_L200,vortex_bessel,compare_proxy_only,planning_proxy,uncalibrated,9.549594,12.850393,287.5,normalised_visualisation
2,3,ell3_core3_L150,vortex_bessel,compare_proxy_only,planning_proxy,uncalibrated,5.876279,6.550601,150.0,normalised_visualisation


C:\PhD\Code\Publication_Study\outputs\csv\materials\material_application_design_table.csv


## Safe Use

The table is safe for design comparison because every row carries
native schema metadata and remains uncalibrated. Experimental
calibration must be joined before making material-response claims.
